# Eurostat Vehicle Registrations – Data Exploration

This dataset provides information about vehicle registrations across European countries. The objective is to understand its structure, historical coverage and suitability for analysing electric vehicle adoption.

In [1]:
# Import libraries
import pandas as pd

# Load the dataset
df_registrations = pd.read_csv("../data/raw/eurostat_vehicle_registrations.csv")

# Inspect the dataset
print("Dataset shape:", df_registrations.shape)

display(df_registrations.head())

df_registrations.info()

Dataset shape: (397, 11)


,DATAFLOW,LAST UPDATE,freq,mot_nrg,vehicle,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:ROAD_EQR_ZEV(1.0),06/10/26 23:00:00,Annual,Electricity,Passenger cars,Number,Albania,2019,24,NaN,NaN
1,ESTAT:ROAD_EQR_ZEV(1.0),06/10/26 23:00:00,Annual,Electricity,Passenger cars,Number,Albania,2020,239,NaN,NaN
2,ESTAT:ROAD_EQR_ZEV(1.0),06/10/26 23:00:00,Annual,Electricity,Passenger cars,Number,Albania,2021,262,NaN,NaN
3,ESTAT:ROAD_EQR_ZEV(1.0),06/10/26 23:00:00,Annual,Electricity,Passenger cars,Number,Albania,2022,624,NaN,NaN
4,ESTAT:ROAD_EQR_ZEV(1.0),06/10/26 23:00:00,Annual,Electricity,Passenger cars,Number,Albania,2023,1671,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 397 entries, 0 to 396
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     397 non-null    str    
 1   LAST UPDATE  397 non-null    str    
 2   freq         397 non-null    str    
 3   mot_nrg      397 non-null    str    
 4   vehicle      397 non-null    str    
 5   unit         397 non-null    str    
 6   geo          397 non-null    str    
 7   TIME_PERIOD  397 non-null    int64  
 8   OBS_VALUE    397 non-null    int64  
 9   OBS_FLAG     51 non-null     str    
 10  CONF_STATUS  0 non-null      float64
dtypes: float64(1), int64(2), str(8)
memory usage: 67.5 KB


In [ ]:
# Check the number of countries and years
print("Number of countries:", df_registrations["geo"].nunique())
print("Countries:", sorted(df_registrations["geo"].unique()))

print("\nYear range:")
print(df_registrations["TIME_PERIOD"].min(), "-", df_registrations["TIME_PERIOD"].max())

# Check categorical variables
for col in ["freq", "mot_nrg", "vehicle", "unit"]:
    print(f"\n{col}:")
    print(df_registrations[col].value_counts())

# Check observation flags
print("\nObservation flags:")
display(df_registrations["OBS_FLAG"].value_counts(dropna=False))

# Check duplicate country-year combinations
print(
    "\nDuplicate country-year combinations:",
    df_registrations.duplicated(subset=["geo", "TIME_PERIOD"]).sum(),
)

Number of countries: 42
Countries: ['Albania', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Estonia', 'European Union - 27 countries (from 2020)', 'Finland', 'France', 'Georgia', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Kosovo*', 'Latvia', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malta', 'Moldova', 'Montenegro', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'Romania', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Türkiye', 'Ukraine', 'United Kingdom']

Year range:
2016 - 2025

freq:
freq
Annual    397
Name: count, dtype: int64

mot_nrg:
mot_nrg
Electricity    397
Name: count, dtype: int64

vehicle:
vehicle
Passenger cars    397
Name: count, dtype: int64

unit:
unit
Number    397
Name: count, dtype: int64

Observation flags:


OBS_FLAG
NaN    346
i       48
e        1
p        1
b        1
Name: count, dtype: int64


Duplicate country-year combinations: 0


In [3]:
# Display the country, year, registration value, and flag for flagged observations
flagged_rows = df_registrations.loc[
    df_registrations["OBS_FLAG"].notna(),
    ["geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"]
]

display(flagged_rows.sort_values(["geo", "TIME_PERIOD"]))

,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG
34,Belgium,2024,127915,i
36,Bulgaria,2016,11,i
37,Bulgaria,2017,13,i
38,Bulgaria,2018,145,i
39,Bulgaria,2019,188,i
40,Bulgaria,2020,139,i
41,Bulgaria,2021,321,i
42,Bulgaria,2022,1021,i
43,Bulgaria,2023,1754,i
44,Bulgaria,2024,1665,i


### Data Quality – Observation Flags

The dataset contains 51 observations with Eurostat flags:
- 48 imputed (`i`)
- 1 estimated (`e`)
- 1 provisional (`p`)
- 1 break in time series (`b`)

These flags indicate special data statuses and should be considered when interpreting results. Flagged observations are retained in the dataset, as removing them could reduce country coverage and bias comparisons.

The dataset contains no duplicate country-year combinations.

In [4]:
# Check missing values in each column
df_registrations.isna().sum()

DATAFLOW         0
LAST UPDATE      0
freq             0
mot_nrg          0
vehicle          0
unit             0
geo              0
TIME_PERIOD      0
OBS_VALUE        0
OBS_FLAG       346
CONF_STATUS    397
dtype: int64

In [6]:
# Check registration values for unusual or invalid entries
print("Negative registrations:", (df_registrations["OBS_VALUE"] < 0).sum())
print("Zero registrations:", (df_registrations["OBS_VALUE"] == 0).sum())

display(df_registrations["OBS_VALUE"].describe())

Negative registrations: 0
Zero registrations: 2


count    3.970000e+02
mean     4.723505e+04
std      1.732856e+05
min      0.000000e+00
25%      3.210000e+02
50%      2.455000e+03
75%      2.187000e+04
max      1.888016e+06
Name: OBS_VALUE, dtype: float64

### Registration Values

No negative registration values were found. Two observations have zero registrations, which are retained as potentially valid values.

Registration counts vary substantially across geographic entities, with larger values likely reflecting aggregate regions such as the EU-27. This variation should be considered when comparing countries.